In [ ]:
from dotenv import load_dotenv


load_dotenv()

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [ ]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)

display(uploader)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "I have some leftover chicken and rice. What can I make?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/jpeg"},
    
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

# print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response)